In [8]:
# ============================================================
# ESR1 VALIDATION AGENT — FINAL PRODUCTION VERSION
# ============================================================
#
# STUDY BACKGROUND:
# A Flatiron Health retrospective study analyzed ESR1
# mutation testing and positivity rates in ER+/HER2-
# metastatic breast cancer patients who initiated 1L
# therapy with aromatase inhibitors or SERDs ±
# CDK4/6 inhibitors (Jan 2020 - Dec 2024)
# Reference: Manuscript in preparation, 2026
#
# KEY FLATIRON FINDINGS:
# → Only 49% of patients underwent ESR1 testing
# → ESR1 positivity at 1L initiation: 25%
# → ESR1 positivity at 2L initiation: 32-34%
# → Mean 1.6 tests per patient
# → 60% tissue-only, 40% blood-only testing
#
# VALIDATION APPROACH:
# This agent independently validates ESR1 mutation
# positivity rates using the MSK Metastatic Breast
# Cancer dataset (Cancer Discovery 2022, n=1116)
# from cBioPortal — a public cancer genomics database
#
# AGENT ARCHITECTURE (5 agents):
# Agent 1 → Data Fetching Agent
#            Connects to cBioPortal API
#            Fetches patient, sample and mutation data
#
# Agent 2 → Analysis Agent
#            Calculates ESR1 positivity rates
#            Builds Table 1 demographics
#            Runs TMB statistical analysis
#            Generates visualizations
#
# Agent 3 → Literature Agent
#            Connects to RWE Research Agent
#            Searches PubMed for published ESR1 rates
#            Extracts key findings from literature
#
# Agent 4 → Comparison Agent
#            Compares cBioPortal findings vs literature
#            Assesses consistency of findings
#            Identifies similarities and differences
#
# Agent 5 → Report Agent
#            Combines all findings
#            Generates structured Word document
#            Includes tables, statistics and plots
#
# VALIDATION HYPOTHESIS:
# If Flatiron findings are generalizable, ESR1
# positivity rates in cBioPortal MSK data should
# be within 5 percentage points of Flatiron (25%)
#
# LIBRARIES:
# pyBioPortal → cBioPortal API access
# pandas      → data manipulation
# scipy       → statistical tests
# matplotlib  → visualizations
# openai      → GPT orchestration
# python-docx → Word report export
# ============================================================

print("=" * 55)
print("ESR1 VALIDATION AGENT")
print("Multi-Agent LLM System for RWE Validation")
print("=" * 55)
print("Manuscript in preparation, 2026")
print("Neha Bansal | Eli Lilly and Company")
print("=" * 55)

ESR1 VALIDATION AGENT
Multi-Agent LLM System for RWE Validation
Manuscript in preparation, 2026
Neha Bansal | Eli Lilly and Company


In [9]:
# ============================================================
# CELL 2 - Installations and Imports
# ============================================================
# All libraries installed and imported in one place
# Run this cell first every time you open this notebook
#
# LIBRARIES:
# openai      → GPT API for agent orchestration
# pybioportal → cBioPortal cancer genomics data access
# pandas      → data manipulation and table building
# scipy       → statistical tests (Mann-Whitney U)
# matplotlib  → visualizations and KM curves
# python-docx → Word document report generation
# biopython   → PubMed API for literature search
# tavily      → web search for additional context
# ============================================================

# Install all required libraries
!pip install openai pybioportal pandas python-docx scipy matplotlib biopython tavily-python -q

# ── Core AI and API libraries ─────────────────────────────
from openai import OpenAI          # GPT agent orchestration
from tavily import TavilyClient    # web search tool
from Bio import Entrez             # PubMed literature search

# ── cBioPortal data access ────────────────────────────────
from pybioportal import clinical_data as cd      # patient/sample data
from pybioportal import mutations as mut          # genomic mutations
from pybioportal import clinical_attributes as ca # available attributes
from pybioportal import studies as st             # study information

# ── Data analysis ─────────────────────────────────────────
import pandas as pd                # data tables and manipulation
import numpy as np                 # numerical calculations
import scipy.stats as sp           # statistical tests

# ── Visualization ─────────────────────────────────────────
import matplotlib.pyplot as plt    # plotting
import matplotlib
matplotlib.use('Agg')              # saves plots as files not popups

# ── Report generation ─────────────────────────────────────
from docx import Document          # Word document creation
from docx.shared import Pt, Inches # Word formatting

# ── Built in Python libraries ─────────────────────────────
import json                        # handle API responses
import getpass                     # safe API key entry
import os                          # file operations
import warnings                    # suppress warnings
from datetime import datetime      # timestamps for filenames
#warnings.filterwarnings('ignore')  # keep output clean

print("✅ All libraries installed and imported!")
print("\nReady for:")
print("→ cBioPortal data access")
print("→ GPT agent orchestration")
print("→ Statistical analysis")
print("→ Visualization")
print("→ Word report generation")


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


✅ All libraries installed and imported!

Ready for:
→ cBioPortal data access
→ GPT agent orchestration
→ Statistical analysis
→ Visualization
→ Word report generation


In [10]:
# ============================================================
# CELL 3 - API Keys and Configuration
# ============================================================
# Load all API keys safely using getpass
# Keys are never saved in the file — only in memory
# Run this cell once at the start of every session
#
# KEYS NEEDED:
# OpenAI  → for GPT agent orchestration
# Tavily  → for web search tool
#
# cBioPortal and PubMed are completely free
# No API key needed for either
#
# CONFIGURATION:
# STUDY_ID and GENE settings are set here as defaults
# but the agent can override them dynamically
# based on your research question
# For example if you ask about PIK3CA instead of ESR1
# the agent will automatically update these values
# ============================================================

# ── OpenAI API Key ────────────────────────────────────────
# Used for GPT agent orchestration
openai_key = getpass.getpass("Paste your OpenAI API key: ")
client = OpenAI(api_key=openai_key)

# ── Tavily API Key ────────────────────────────────────────
# Used for web search tool
tavily_key = getpass.getpass("Paste your Tavily API key: ")
tavily_client = TavilyClient(api_key=tavily_key)

# ── PubMed Configuration ──────────────────────────────────
PUBMED_EMAIL = "bansalneha2511@gmail.com"
Entrez.email = PUBMED_EMAIL

# ── Default Study and Gene Configuration ──────────────────
# These are DEFAULT values for our ESR1 validation study
# The agent can override these dynamically
# based on the research question asked
#
# To use a different study → change STUDY_ID
# To use a different gene  → change GENE_SYMBOL
#                            and GENE_ENTREZ_ID
#
# Common breast cancer genes and their Entrez IDs:
# ESR1    → 2099   (estrogen receptor)
# PIK3CA  → 5290   (PI3K pathway)
# TP53    → 7157   (tumor suppressor)
# CDH1    → 999    (lobular breast cancer)
# BRCA1   → 672    (hereditary breast cancer)
# BRCA2   → 675    (hereditary breast cancer)
#
# Common cBioPortal breast cancer study IDs:
# breast_ink4_msk_2021  → MSK MBC Cancer Discovery 2022
# brca_tcga_pan_can_atlas_2018 → TCGA PanCancer Atlas
# breast_msk_2018       → MSK Cancer Cell 2018
STUDY_ID = "breast_ink4_msk_2021"
GENE_SYMBOL = "ESR1"
GENE_ENTREZ_ID = 2099

print("✅ All API keys loaded and configuration set!")
print(f"\nDefault configuration:")
print(f"→ Study:   {STUDY_ID}")
print(f"→ Gene:    {GENE_SYMBOL}")
print(f"→ Entrez ID: {GENE_ENTREZ_ID}")
print(f"\nNote: Agent can override these dynamically")
print(f"      based on your research question")

✅ All API keys loaded and configuration set!

Default configuration:
→ Study:   breast_ink4_msk_2021
→ Gene:    ESR1
→ Entrez ID: 2099

Note: Agent can override these dynamically
      based on your research question


In [11]:
# ============================================================
# CELL 4 - Data Fetching Functions
# ============================================================
# All cBioPortal data fetching wrapped as clean functions
# Each function does ONE specific job
# Takes study_id and gene parameters as inputs
# Returns clean pandas DataFrames as outputs
#
# WHY FUNCTIONS INSTEAD OF LOOSE CODE:
# Functions are reusable — call them for any study or gene
# Functions are testable — easy to verify each one works
# Functions are agent-ready — GPT can call them as tools
#
# THREE FUNCTIONS:
# fetch_patient_data()   → demographics table
# fetch_sample_data()    → tumor characteristics table
# fetch_gene_mutations() → mutation data for any gene
# ============================================================

def fetch_patient_data(study_id):
    """
    Fetches patient level clinical data from cBioPortal.
    Returns one row per patient with demographics.
    
    Args:
        study_id → cBioPortal study identifier
                   e.g. "breast_ink4_msk_2021"
    
    Returns:
        pandas DataFrame with patient demographics
        columns: patientId, AGE_CURRENT, SEX, RACE, ETHNICITY
    """
    print(f"Fetching patient data for study: {study_id}...")
    
    # Fetch patient level clinical data
    # ret_format=WIDE → one row per patient
    # clinical_data_type=PATIENT → demographics not tumor data
    patient_data = cd.fetch_all_clinical_data_in_study(
        study_id=study_id,
        clinical_data_type="PATIENT",
        ret_format="WIDE"
    )
    
    print(f"✅ Patient data fetched: {len(patient_data)} patients")
    return patient_data


def fetch_sample_data(study_id):
    """
    Fetches sample level clinical data from cBioPortal.
    Returns one row per tumor sample with tumor characteristics.
    
    Args:
        study_id → cBioPortal study identifier
    
    Returns:
        pandas DataFrame with tumor characteristics
        columns: patientId, sampleId, METASTATIC_SITE,
                 MSI_TYPE, TMB_NONSYNONYMOUS, CANCER_TYPE_DETAILED
    """
    print(f"Fetching sample data for study: {study_id}...")
    
    # Fetch sample level clinical data
    # clinical_data_type=SAMPLE → tumor data not demographics
    sample_data = cd.fetch_all_clinical_data_in_study(
        study_id=study_id,
        clinical_data_type="SAMPLE",
        ret_format="WIDE"
    )
    
    # Keep one sample per patient
    # Some patients have multiple biopsies
    # We deduplicate to avoid counting patients twice
    sample_data_dedup = sample_data.drop_duplicates(
        subset='patientId',
        keep='first'
    )
    
    print(f"✅ Sample data fetched: {len(sample_data_dedup)} unique patients")
    return sample_data_dedup


def fetch_gene_mutations(study_id, entrez_gene_id, gene_symbol):
    """
    Fetches mutation data for a specific gene from cBioPortal.
    Returns all mutations for that gene in the study.
    
    Args:
        study_id       → cBioPortal study identifier
        entrez_gene_id → unique gene ID number
                         ESR1=2099, PIK3CA=5290, TP53=7157
        gene_symbol    → gene name for display
                         e.g. "ESR1", "PIK3CA"
    
    Returns:
        pandas DataFrame with mutation details
        columns: patientId, sampleId, proteinChange,
                 mutationType, mutationStatus
    """
    print(f"Fetching {gene_symbol} mutations for study: {study_id}...")
    
    # Fetch mutations for specific gene
    # molecular_profile_id = study_id + "_mutations"
    # sample_list_id = study_id + "_all" means all samples
    gene_mutations = mut.get_muts_in_mol_prof_by_sample_list_id(
        molecular_profile_id=f"{study_id}_mutations",
        sample_list_id=f"{study_id}_all",
        entrez_gene_id=entrez_gene_id
    )
    
    print(f"✅ {gene_symbol} mutations fetched: {len(gene_mutations)} mutation events")
    print(f"   Unique patients with {gene_symbol}: {gene_mutations['patientId'].nunique()}")
    return gene_mutations


# ── TEST ALL THREE FUNCTIONS ──────────────────────────────
# Run all three functions with our default configuration
# Verify everything works before building agent on top
print("=" * 55)
print("TESTING DATA FETCHING FUNCTIONS")
print("=" * 55)

# Fetch all three datasets
patient_data = fetch_patient_data(STUDY_ID)
sample_data = fetch_sample_data(STUDY_ID)
gene_mutations = fetch_gene_mutations(
    STUDY_ID, 
    GENE_ENTREZ_ID, 
    GENE_SYMBOL
)

print("\n✅ All data fetching functions working!")
print(f"\nSummary:")
print(f"→ Patients:   {len(patient_data)}")
print(f"→ Samples:    {len(sample_data)}")
print(f"→ Mutations:  {len(gene_mutations)}")

TESTING DATA FETCHING FUNCTIONS
Fetching patient data for study: breast_ink4_msk_2021...
✅ Patient data fetched: 1116 patients
Fetching sample data for study: breast_ink4_msk_2021...
✅ Sample data fetched: 1116 unique patients
Fetching ESR1 mutations for study: breast_ink4_msk_2021...
✅ ESR1 mutations fetched: 305 mutation events
   Unique patients with ESR1: 238

✅ All data fetching functions working!

Summary:
→ Patients:   1116
→ Samples:    1116
→ Mutations:  305


In [12]:
# ============================================================
# CELL 5 - Analysis Functions
# ============================================================
# All analysis code wrapped as clean reusable functions
# Each function takes DataFrames as input
# and returns results as dictionaries or DataFrames
#
# FIVE FUNCTIONS:
# build_merged_dataset()   → combines all three tables
# calculate_mutation_rate() → calculates positivity rate
# generate_table1()        → demographic summary table
# run_tmb_analysis()       → TMB statistical comparison
# generate_tmb_plot()      → TMB visualization
# ============================================================

# ── HELPER FUNCTIONS ──────────────────────────────────────

def median_iqr(series):
    """Calculate median and IQR for continuous variables"""
    numeric = pd.to_numeric(series, errors='coerce')
    median = numeric.median()
    q1 = numeric.quantile(0.25)
    q3 = numeric.quantile(0.75)
    return f"{median:.1f} ({q1:.1f} - {q3:.1f})"

def n_pct(series, value):
    """Calculate n and % for categorical variables"""
    count = (series == value).sum()
    pct = round((count / len(series)) * 100, 1)
    return f"{count} ({pct}%)"

def group_race(race):
    """Groups race into main categories"""
    if race == 'WHITE':
        return 'White'
    elif race == 'BLACK OR AFRICAN AMERICAN':
        return 'Black or African American'
    elif race == 'ASIAN-FAR EAST/INDIAN SUBCONT':
        return 'Asian'
    else:
        return 'Other/Unknown'

def group_ethnicity(eth):
    """Groups ethnicity into Hispanic vs Non-Hispanic"""
    if 'Non-Spanish' in str(eth):
        return 'Non-Hispanic'
    elif eth == 'Unknown whether Spanish or not':
        return 'Unknown'
    else:
        return 'Hispanic'

def group_cancer_type(cancer):
    """Groups cancer types into main categories"""
    if 'Ductal' in str(cancer):
        return 'Invasive Ductal'
    elif 'Lobular' in str(cancer):
        return 'Invasive Lobular'
    else:
        return 'Other'

# ── MAIN ANALYSIS FUNCTIONS ───────────────────────────────

def build_merged_dataset(patient_data, sample_data, gene_mutations, gene_symbol):
    """
    Merges patient, sample and mutation tables into
    one complete dataset with gene mutation status column.
    
    Args:
        patient_data   → from fetch_patient_data()
        sample_data    → from fetch_sample_data()
        gene_mutations → from fetch_gene_mutations()
        gene_symbol    → gene name e.g. "ESR1"
    
    Returns:
        merged pandas DataFrame with all columns
        plus MUTATION_STATUS column (Positive/Negative)
    """
    print(f"Building merged dataset...")
    
    # Merge patient + sample data on patientId
    # LEFT JOIN keeps all patients even without sample data
    merged = patient_data.merge(
        sample_data[[
            'patientId', 'METASTATIC_SITE', 'MSI_TYPE',
            'TMB_NONSYNONYMOUS', 'CANCER_TYPE_DETAILED',
            'SAMPLE_TYPE', 'MUTATION_COUNT'
        ]],
        on='patientId',
        how='left'
    )
    
    # Add grouped demographic columns
    merged['RACE_GROUPED'] = merged['RACE'].apply(group_race)
    merged['ETHNICITY_GROUPED'] = merged['ETHNICITY'].apply(group_ethnicity)
    merged['CANCER_TYPE_GROUPED'] = merged['CANCER_TYPE_DETAILED'].apply(
        group_cancer_type
    )
    
    # Add mutation status column
    # Positive if patientId appears in gene_mutations
    # Negative if patientId does not appear
    mutated_patients = gene_mutations['patientId'].unique()
    status_col = f"{gene_symbol}_STATUS"
    merged[status_col] = merged['patientId'].apply(
        lambda x: f'{gene_symbol} Positive' 
        if x in mutated_patients 
        else f'{gene_symbol} Negative'
    )
    
    # Convert TMB to numeric
    merged['TMB_NUMERIC'] = pd.to_numeric(
        merged['TMB_NONSYNONYMOUS'], errors='coerce'
    )
    
    print(f"✅ Merged dataset built: {len(merged)} patients")
    print(f"   {gene_symbol} Positive: {(merged[status_col] == f'{gene_symbol} Positive').sum()}")
    print(f"   {gene_symbol} Negative: {(merged[status_col] == f'{gene_symbol} Negative').sum()}")
    return merged


def calculate_mutation_rate(merged_data, gene_mutations, gene_symbol):
    """
    Calculates mutation positivity rate for a gene.
    
    Args:
        merged_data    → from build_merged_dataset()
        gene_mutations → from fetch_gene_mutations()
        gene_symbol    → gene name e.g. "ESR1"
    
    Returns:
        dictionary with positivity rate and counts
    """
    print(f"Calculating {gene_symbol} positivity rate...")
    
    total_patients = len(merged_data)
    patients_with_mutation = gene_mutations['patientId'].nunique()
    patients_without_mutation = total_patients - patients_with_mutation
    positivity_rate = round(
        (patients_with_mutation / total_patients) * 100, 1
    )
    
    results = {
        "gene": gene_symbol,
        "total_patients": total_patients,
        "patients_positive": patients_with_mutation,
        "patients_negative": patients_without_mutation,
        "positivity_rate": positivity_rate,
        "total_mutation_events": len(gene_mutations),
        "study": STUDY_ID,
        "database": "cBioPortal"
    }
    
    print(f"✅ {gene_symbol} positivity rate: {positivity_rate}%")
    return results


def generate_table1(merged_data, gene_symbol):
    """
    Generates Table 1 patient characteristics
    overall and stratified by mutation status.
    
    Args:
        merged_data  → from build_merged_dataset()
        gene_symbol  → gene name e.g. "ESR1"
    
    Returns:
        prints formatted Table 1
        returns dictionary of key demographics
    """
    print(f"Generating Table 1...")
    
    status_col = f"{gene_symbol}_STATUS"
    overall = merged_data
    mut_pos = merged_data[
        merged_data[status_col] == f'{gene_symbol} Positive'
    ]
    mut_neg = merged_data[
        merged_data[status_col] == f'{gene_symbol} Negative'
    ]
    
    n_overall = len(overall)
    n_pos = len(mut_pos)
    n_neg = len(mut_neg)
    
    print("\n" + "=" * 75)
    print(f"{'TABLE 1: PATIENT CHARACTERISTICS':^75}")
    print("=" * 75)
    print(f"{'Characteristic':<35} {'Overall':^15} {gene_symbol+'+':^12} {gene_symbol+'-':^12}")
    print(f"{'':35} {f'(N={n_overall})':^15} {f'(N={n_pos})':^12} {f'(N={n_neg})':^12}")
    print("-" * 75)

    # Age
    print(f"\n{'Age — median (IQR)':<35}")
    print(f"  {'Overall':<33} {median_iqr(overall['AGE_CURRENT'])}")
    print(f"  {gene_symbol+' Positive':<33} {median_iqr(mut_pos['AGE_CURRENT'])}")
    print(f"  {gene_symbol+' Negative':<33} {median_iqr(mut_neg['AGE_CURRENT'])}")

    # Sex
    print(f"\n{'Sex':<35}")
    print(f"  {'Female — n (%)':<33}", end="")
    print(f"{n_pct(overall['SEX'], 'Female'):^15}", end="")
    print(f"{n_pct(mut_pos['SEX'], 'Female'):^12}", end="")
    print(f"{n_pct(mut_neg['SEX'], 'Female'):^12}")

    # Race
    print(f"\n{'Race':<35}")
    for cat in ['White', 'Black or African American', 'Asian', 'Other/Unknown']:
        print(f"  {cat+' — n (%)':<33}", end="")
        print(f"{n_pct(overall['RACE_GROUPED'], cat):^15}", end="")
        print(f"{n_pct(mut_pos['RACE_GROUPED'], cat):^12}", end="")
        print(f"{n_pct(mut_neg['RACE_GROUPED'], cat):^12}")

    # Ethnicity
    print(f"\n{'Ethnicity':<35}")
    for cat in ['Non-Hispanic', 'Hispanic', 'Unknown']:
        print(f"  {cat+' — n (%)':<33}", end="")
        print(f"{n_pct(overall['ETHNICITY_GROUPED'], cat):^15}", end="")
        print(f"{n_pct(mut_pos['ETHNICITY_GROUPED'], cat):^12}", end="")
        print(f"{n_pct(mut_neg['ETHNICITY_GROUPED'], cat):^12}")

    # Cancer Type
    print(f"\n{'Cancer Type':<35}")
    for cat in ['Invasive Ductal', 'Invasive Lobular', 'Other']:
        print(f"  {cat+' — n (%)':<33}", end="")
        print(f"{n_pct(overall['CANCER_TYPE_GROUPED'], cat):^15}", end="")
        print(f"{n_pct(mut_pos['CANCER_TYPE_GROUPED'], cat):^12}", end="")
        print(f"{n_pct(mut_neg['CANCER_TYPE_GROUPED'], cat):^12}")

    # Metastatic Site
    print(f"\n{'Metastatic Site (top 5)':<35}")
    for site in ['Liver', 'Bone', 'Lymph Node', 'Lung', 'Skin']:
        print(f"  {site+' — n (%)':<33}", end="")
        print(f"{n_pct(overall['METASTATIC_SITE'], site):^15}", end="")
        print(f"{n_pct(mut_pos['METASTATIC_SITE'], site):^12}", end="")
        print(f"{n_pct(mut_neg['METASTATIC_SITE'], site):^12}")

    # MSI Status
    print(f"\n{'MSI Status':<35}")
    for cat in ['Stable', 'Indeterminate', 'Instable']:
        print(f"  {cat+' — n (%)':<33}", end="")
        print(f"{n_pct(overall['MSI_TYPE'], cat):^15}", end="")
        print(f"{n_pct(mut_pos['MSI_TYPE'], cat):^12}", end="")
        print(f"{n_pct(mut_neg['MSI_TYPE'], cat):^12}")

    # TMB
    print(f"\n{'TMB — median (IQR)':<35}")
    print(f"  {'Overall':<33} {median_iqr(overall['TMB_NUMERIC'])}")
    print(f"  {gene_symbol+' Positive':<33} {median_iqr(mut_pos['TMB_NUMERIC'])}")
    print(f"  {gene_symbol+' Negative':<33} {median_iqr(mut_neg['TMB_NUMERIC'])}")

    print("\n" + "=" * 75)

    # Store key results
    table1_results = {
        "total_patients": n_overall,
        "mutation_positive": n_pos,
        "mutation_negative": n_neg,
        "median_age": pd.to_numeric(
            overall['AGE_CURRENT'], errors='coerce'
        ).median(),
        "pct_female": round(
            (overall['SEX'] == 'Female').sum() / n_overall * 100, 1
        ),
        "pct_white": round(
            (overall['RACE_GROUPED'] == 'White').sum() / n_overall * 100, 1
        ),
        "top_metastatic_site": overall['METASTATIC_SITE'].value_counts().index[0]
    }

    print(f"✅ Table 1 generated!")
    return table1_results


def run_tmb_analysis(merged_data, gene_symbol):
    """
    Runs TMB statistical comparison between
    mutation positive and negative patients.
    Uses Mann-Whitney U test (non-parametric).
    
    Args:
        merged_data → from build_merged_dataset()
        gene_symbol → gene name e.g. "ESR1"
    
    Returns:
        dictionary with TMB statistics and p-value
    """
    print(f"Running TMB analysis...")
    
    status_col = f"{gene_symbol}_STATUS"
    
    tmb_overall = merged_data['TMB_NUMERIC'].dropna()
    tmb_pos = merged_data[
        merged_data[status_col] == f'{gene_symbol} Positive'
    ]['TMB_NUMERIC'].dropna()
    tmb_neg = merged_data[
        merged_data[status_col] == f'{gene_symbol} Negative'
    ]['TMB_NUMERIC'].dropna()
    
    # Mann-Whitney U test
    statistic, p_value = sp.mannwhitneyu(
        tmb_pos, tmb_neg, alternative='two-sided'
    )
    p_value_rounded = round(p_value, 4)
    
    tmb_results = {
        "overall_median": round(float(tmb_overall.median()), 2),
        "overall_iqr": f"{round(float(tmb_overall.quantile(0.25)), 2)}-{round(float(tmb_overall.quantile(0.75)), 2)}",
        "positive_median": round(float(tmb_pos.median()), 2),
        "positive_iqr": f"{round(float(tmb_pos.quantile(0.25)), 2)}-{round(float(tmb_pos.quantile(0.75)), 2)}",
        "negative_median": round(float(tmb_neg.median()), 2),
        "negative_iqr": f"{round(float(tmb_neg.quantile(0.25)), 2)}-{round(float(tmb_neg.quantile(0.75)), 2)}",
        "p_value": float(p_value_rounded),
        "significant": bool(p_value < 0.05),
        "test": "Mann-Whitney U"
    }
    
    print(f"✅ TMB analysis complete!")
    print(f"   {gene_symbol}+ median TMB: {tmb_results['positive_median']}")
    print(f"   {gene_symbol}- median TMB: {tmb_results['negative_median']}")
    print(f"   P-value: {p_value_rounded}")
    return tmb_results, tmb_pos, tmb_neg, tmb_overall


def generate_tmb_plot(tmb_overall, tmb_pos, tmb_neg, 
                      gene_symbol, p_value, study_id):
    """
    Generates TMB boxplot comparing mutation positive
    vs negative patients. Saves as PNG file.
    
    Args:
        tmb_overall → overall TMB values
        tmb_pos     → TMB values for mutation positive
        tmb_neg     → TMB values for mutation negative
        gene_symbol → gene name e.g. "ESR1"
        p_value     → from run_tmb_analysis()
        study_id    → for plot title
    
    Returns:
        filename of saved plot
    """
    print(f"Generating TMB plot...")
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    bp = ax.boxplot(
        [tmb_overall.tolist(), tmb_pos.tolist(), tmb_neg.tolist()],
        patch_artist=True,
        labels=[
            f'Overall\n(N={len(tmb_overall)})',
            f'{gene_symbol}+\n(N={len(tmb_pos)})',
            f'{gene_symbol}-\n(N={len(tmb_neg)})'
        ],
        medianprops=dict(color='black', linewidth=2),
        flierprops=dict(marker='o', markersize=3, alpha=0.5)
    )
    
    colors = ['#AED6F1', '#E74C3C', '#2ECC71']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax.annotate(
        f'p = {p_value}\n(Mann-Whitney U)',
        xy=(0.98, 0.95),
        xycoords='axes fraction',
        ha='right', va='top', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    )
    
    medians = [
        round(pd.Series(tmb_overall.tolist()).median(), 2),
        round(pd.Series(tmb_pos.tolist()).median(), 2),
        round(pd.Series(tmb_neg.tolist()).median(), 2)
    ]
    for i, median in enumerate(medians):
        ax.text(i + 1, median + 0.2, f'{median}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    ax.set_title(
        f'Tumor Mutation Burden (TMB) by {gene_symbol} Mutation Status\n'
        f'MSK Metastatic Breast Cancer (Cancer Discovery 2022)',
        fontsize=13, fontweight='bold', pad=15
    )
    ax.set_ylabel('TMB (Nonsynonymous mutations/Mb)', fontsize=12)
    ax.set_xlabel(f'{gene_symbol} Mutation Status', fontsize=12)
    ax.set_ylim(0, 40)
    ax.yaxis.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    
    plot_filename = f"tmb_{gene_symbol}_comparison.png"
    plt.savefig(plot_filename, dpi=150, 
                bbox_inches='tight', facecolor='white')
    plt.close()
    
    print(f"✅ TMB plot saved as: {plot_filename}")
    return plot_filename


# ── TEST ALL ANALYSIS FUNCTIONS ───────────────────────────
print("=" * 55)
print("TESTING ANALYSIS FUNCTIONS")
print("=" * 55)

# Build merged dataset
merged_data = build_merged_dataset(
    patient_data, sample_data, gene_mutations, GENE_SYMBOL
)

# Calculate mutation rate
mutation_rate_results = calculate_mutation_rate(
    merged_data, gene_mutations, GENE_SYMBOL
)

# Generate Table 1
table1_results = generate_table1(merged_data, GENE_SYMBOL)

# Run TMB analysis
tmb_results, tmb_pos, tmb_neg, tmb_overall = run_tmb_analysis(
    merged_data, GENE_SYMBOL
)

# Generate TMB plot
plot_file = generate_tmb_plot(
    tmb_overall, tmb_pos, tmb_neg,
    GENE_SYMBOL, tmb_results['p_value'], STUDY_ID
)

print("\n✅ All analysis functions working!")

TESTING ANALYSIS FUNCTIONS
Building merged dataset...
✅ Merged dataset built: 1116 patients
   ESR1 Positive: 238
   ESR1 Negative: 878
Calculating ESR1 positivity rate...
✅ ESR1 positivity rate: 21.3%
Generating Table 1...

                     TABLE 1: PATIENT CHARACTERISTICS                      
Characteristic                          Overall        ESR1+        ESR1-    
                                       (N=1116)       (N=238)      (N=878)   
---------------------------------------------------------------------------

Age — median (IQR)                 
  Overall                           61.0 (53.0 - 70.0)
  ESR1 Positive                     62.0 (54.0 - 70.0)
  ESR1 Negative                     61.0 (53.0 - 70.0)

Sex                                
  Female — n (%)                    1093 (97.9%)  236 (99.2%) 857 (97.6%) 

Race                               
  White — n (%)                      878 (78.7%)  200 (84.0%) 678 (77.2%) 
  Black or African American — n (%)   87 

C:\Users\bansa\AppData\Local\Temp\ipykernel_24232\3444564384.py:345: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(


In [2]:
# ============================================================
# CELL 6 - Tool Descriptions for GPT
# ============================================================
# This tells GPT what tools are available
# and when to use each one
#
# We have 5 tools:
# 1. fetch_cbioportal_data  → gets cancer genomics data
# 2. analyze_mutations      → calculates rates and stats
# 3. search_pubmed          → finds published literature
# 4. search_web             → gets current information
# 5. generate_report        → creates Word document
#
# HOW GPT USES THESE:
# GPT reads the descriptions and decides which tool
# to call based on your research question
# It passes the right parameters automatically
# Your code runs the tool and returns results
# GPT reads results and decides next step
#
# THIS IS THE AGENT PATTERN:
# Tool descriptions → GPT decides → code runs → 
# results returned → GPT decides again → loop
# ============================================================

tools = [
    {
        "type": "function",
        "function": {
            "name": "fetch_cbioportal_data",
            "description": """Use this tool to fetch cancer genomics data 
            from cBioPortal for a specific gene and study.
            This fetches patient demographics, sample characteristics,
            and gene mutation data from the specified study.
            Use this as the FIRST step in any validation analysis.
            Returns patient count, mutation events and merged dataset.""",
            "parameters": {
                "type": "object",
                "properties": {
                    "study_id": {
                        "type": "string",
                        "description": "cBioPortal study ID e.g. breast_ink4_msk_2021"
                    },
                    "gene_symbol": {
                        "type": "string",
                        "description": "Gene name e.g. ESR1, PIK3CA, TP53"
                    },
                    "entrez_gene_id": {
                        "type": "integer",
                        "description": "Entrez gene ID. ESR1=2099, PIK3CA=5290, TP53=7157"
                    }
                },
                "required": ["study_id", "gene_symbol", "entrez_gene_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "analyze_mutations",
            "description": """Use this tool AFTER fetch_cbioportal_data.
            Calculates mutation positivity rate, generates Table 1
            demographics stratified by mutation status, runs TMB
            statistical analysis and generates TMB visualization.
            Returns positivity rate, table1 summary and TMB results.""",
            "parameters": {
                "type": "object",
                "properties": {
                    "gene_symbol": {
                        "type": "string",
                        "description": "Gene name e.g. ESR1, PIK3CA"
                    }
                },
                "required": ["gene_symbol"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_pubmed",
            "description": """Use this tool to search PubMed for published
            research papers on any medical or scientific topic.
            Use for finding published mutation rates, clinical findings,
            treatment outcomes or any healthcare research question.
            Always use this instead of web search for medical topics.
            Returns real verified paper titles, authors and findings.""",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Medical research search term e.g. ESR1 mutation metastatic breast cancer real world"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_memory",
            "description": """Search the agent's PERSISTENT MEMORY — a vector
            database of papers collected in previous runs — for papers
            relevant to a topic. ALWAYS call this BEFORE search_pubmed.
            It is fast and free because it reuses papers already gathered.
            Read how many relevant papers it returns to decide how many
            PubMed searches you still need. If it finds nothing relevant,
            call search_pubmed to fetch fresh papers.""",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Medical search term, same style as search_pubmed e.g. ESR1 mutation metastatic breast cancer"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": """Use this tool to search the web for current
            general information. Use ONLY for non-medical queries.
            For any medical or research topics always use search_pubmed.""",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "General search query"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_report",
            "description": """Use this tool as the FINAL step after all
            analysis is complete. Generates a structured Word document
            combining all findings including positivity rates,
            Table 1 demographics, TMB analysis, TMB plot and
            comparison against published literature.
            Always call this last after all other tools have run.""",
            "parameters": {
                "type": "object",
                "properties": {
                    "research_question": {
                        "type": "string",
                        "description": "The original research question asked"
                    }
                },
                "required": ["research_question"]
            }
        }
    }
]

print("✅ Tool descriptions ready!")
print(f"\n{len(tools)} tools available for GPT:")
for tool in tools:
    print(f"→ {tool['function']['name']}")

✅ Tool descriptions ready!

6 tools available for GPT:
→ fetch_cbioportal_data
→ analyze_mutations
→ search_pubmed
→ search_memory
→ search_web
→ generate_report


In [4]:
# ============================================================
# CELL 7 - Tool Execution Functions (Updated)
# ============================================================
# These functions actually RUN when GPT calls a tool
# Think of Cell 6 as the menu
# and Cell 7 as the kitchen that prepares the food
#
# SHARED WHITEBOARD (Global Variables):
# These store results between agent steps
# Each agent reads from and writes to this whiteboard
# Without these boxes results disappear after each step
#
# AGENT 1 → run_fetch_cbioportal_data()
#            writes: _patient_data, _sample_data
#                    _gene_mutations, _merged_data
#
# AGENT 2 → run_analyze_mutations()
#            reads:  _merged_data, _gene_mutations
#            writes: _mutation_rate_results
#                    _table1_results
#                    _tmb_results, _plot_file
#
# AGENT 3 → run_search_pubmed()
#            writes: _pubmed_results
#            FIX: now saves to whiteboard
#                 previously results were lost
#
# AGENT 4 → run_search_web()
#            no whiteboard needed
#            results go directly to GPT
#
# AGENT 5 → run_generate_report()
#            reads:  ALL whiteboard results
#            writes: Word document file
#            FIX: now includes PubMed comparison
# ============================================================

# ── SHARED WHITEBOARD ─────────────────────────────────────
# All variables start as None (empty boxes)
# Each agent fills its relevant boxes when it runs
# None means the box is empty — nothing stored yet

_patient_data = None           # patient demographics table
_sample_data = None            # tumor characteristics table
_gene_mutations = None         # gene mutation data
_merged_data = None            # combined table with mutation status
_mutation_rate_results = None  # positivity rate and counts
_table1_results = None         # demographic summary
_tmb_results = None            # TMB statistics and p-value
_plot_file = None              # path to saved TMB plot
_pubmed_results = None         # ← NEW: PubMed literature results
                               #   previously missing causing
                               #   report to have no literature section


# ── AGENT 1: FETCH CBIOPORTAL DATA ────────────────────────
def run_fetch_cbioportal_data(study_id, gene_symbol, entrez_gene_id):
    """
    Agent 1 - Fetches all data from cBioPortal
    Runs the three fetching functions from Cell 4
    Stores results on shared whiteboard
    Must run BEFORE analyze_mutations
    
    Args:
        study_id       → cBioPortal study identifier
        gene_symbol    → gene name e.g. ESR1
        entrez_gene_id → unique gene number e.g. 2099
    
    Returns:
        JSON summary for GPT to read
    """
    # Declare which whiteboard boxes we will write to
    # Without 'global' Python creates local copies
    # that disappear when function ends
    global _patient_data, _sample_data, _gene_mutations, _merged_data
    
    print(f"\n[Agent 1] Fetching cBioPortal data...")
    print(f"[Agent 1] Study: {study_id}")
    print(f"[Agent 1] Gene: {gene_symbol}")
    
    # Run the three fetching functions from Cell 4
    # Results stored directly on whiteboard
    _patient_data = fetch_patient_data(study_id)
    _sample_data = fetch_sample_data(study_id)
    _gene_mutations = fetch_gene_mutations(
        study_id, entrez_gene_id, gene_symbol
    )
    
    # Build merged dataset immediately
    # Combines all three tables into one
    _merged_data = build_merged_dataset(
        _patient_data, _sample_data,
        _gene_mutations, gene_symbol
    )
    
    # Return summary to GPT
    # json.dumps converts Python dictionary to text
    # GPT reads this text and decides next step
    return json.dumps({
        "status": "success",
        "total_patients": len(_patient_data),
        "total_mutations": len(_gene_mutations),
        "patients_with_mutation": int(
            _gene_mutations['patientId'].nunique()
        ),
        "study": study_id,
        "gene": gene_symbol,
        "message": "Data fetched successfully. Ready for analysis."
    })


# ── AGENT 2: ANALYZE MUTATIONS ────────────────────────────
def run_analyze_mutations(gene_symbol):
    """
    Agent 2 - Runs all analysis on fetched data
    Calculates positivity rate, Table 1, TMB analysis
    Must run AFTER fetch_cbioportal_data
    
    Args:
        gene_symbol → gene name e.g. ESR1
    
    Returns:
        JSON summary with key statistics for GPT
    """
    # Declare which whiteboard boxes we will write to
    global _mutation_rate_results, _table1_results
    global _tmb_results, _plot_file

    # Safety check — verify Agent 1 ran first
    # If _merged_data is still None Agent 1 never ran
    # Return helpful error instead of crashing
    if _merged_data is None:
        return json.dumps({
            "status": "error",
            "message": "Please run fetch_cbioportal_data first"
        })
    
    print(f"\n[Agent 2] Running mutation analysis for {gene_symbol}...")
    
    # Calculate mutation positivity rate
    # Stores result in _mutation_rate_results box
    _mutation_rate_results = calculate_mutation_rate(
        _merged_data, _gene_mutations, gene_symbol
    )
    
    # Generate Table 1 demographics
    # Stores result in _table1_results box
    _table1_results = generate_table1(_merged_data, gene_symbol)
    
    # Run TMB statistical analysis
    # Returns 4 values — we unpack them separately
    # tmb_data    → statistics dictionary
    # tmb_pos     → TMB values for mutation positive patients
    # tmb_neg     → TMB values for mutation negative patients
    # tmb_overall → TMB values for all patients
    tmb_data, tmb_pos, tmb_neg, tmb_overall = run_tmb_analysis(
        _merged_data, gene_symbol
    )
    _tmb_results = tmb_data
    
    # Generate TMB boxplot
    # Saves PNG file and returns filename
    _plot_file = generate_tmb_plot(
        tmb_overall, tmb_pos, tmb_neg,
        gene_symbol, tmb_data['p_value'], STUDY_ID
    )
    
    # Return summary to GPT
    return json.dumps({
        "status": "success",
        "positivity_rate": _mutation_rate_results['positivity_rate'],
        "total_patients": _mutation_rate_results['total_patients'],
        "patients_positive": _mutation_rate_results['patients_positive'],
        "tmb_positive_median": _tmb_results['positive_median'],
        "tmb_negative_median": _tmb_results['negative_median'],
        "tmb_p_value": _tmb_results['p_value'],
        "tmb_significant": _tmb_results['significant'],
        "plot_saved": _plot_file,
        "message": f"Analysis complete. {gene_symbol} positivity: "
                   f"{_mutation_rate_results['positivity_rate']}%"
    })


# ── AGENT 3: SEARCH PUBMED ────────────────────────────────
# ── AGENT 3: SEARCH PUBMED ────────────────────────────────
def run_search_pubmed(query):
    """
    Agent 3 - Searches PubMed for published literature
    Fetches structured XML data for clean parsing
    Extracts title, authors, year, abstract for each paper
    Saves structured results to whiteboard for report agent
    
    FIX FROM ORIGINAL:
    - Changed from plain text to XML for structured parsing
    - Now extracts title, authors, year, abstract separately
    - Results saved as list of dictionaries not raw text blob
    - Report agent can now build proper table from results
    
    Args:
        query - medical research search term
    
    Returns:
        JSON with structured paper details for GPT
    """
    global _pubmed_results

    print(f"\n[Agent 3] Searching PubMed: {query}")

    # ── STEP 1: Search for matching paper IDs ─────────────
    # esearch returns a list of PubMed IDs matching the query
    # retmax=10 means return maximum 10 paper IDs
    search_handle = Entrez.esearch(
        db="pubmed",
        term=query,
        retmax=10        # fetch 10 papers for the table
    )
    search_results = Entrez.read(search_handle)
    search_handle.close()

    paper_ids = search_results["IdList"]

    # If no papers found return clear message
    if not paper_ids:
        return json.dumps({
            "status": "no_results",
            "message": f"No papers found for: {query}"
        })

    # ── STEP 2: Fetch full paper details in XML format ────
    # rettype="xml" gives structured data with separate fields
    # retmode="xml" ensures we get proper XML back
    # This replaces the old rettype="abstract" retmode="text"
    fetch_handle = Entrez.efetch(
        db="pubmed",
        id=paper_ids,
        rettype="xml",   # structured XML not plain text
        retmode="xml"    # proper XML format
    )

    # Entrez.read parses the XML automatically into
    # a Python dictionary we can navigate cleanly
    records = Entrez.read(fetch_handle)
    fetch_handle.close()

    # ── STEP 3: Extract structured fields from each paper ─
    # Loop through each paper and pull out the fields we need
    # Store as a list of clean dictionaries
    papers_list = []

    for record in records["PubmedArticle"]:

        # Navigate XML structure to get each field
        # Each field is nested inside MedlineCitation
        article = record["MedlineCitation"]["Article"]

        # Title — straightforward string field
        title = str(article.get("ArticleTitle", "No title available"))

        # Authors — list of author objects, we join them
        # Each author has LastName and ForeName
        # We take first 3 authors and add "et al." if more
        authors = "Unknown"
        if "AuthorList" in article:
            author_list = article["AuthorList"]
            author_names = []
            for author in author_list[:3]:  # first 3 authors only
                last = author.get("LastName", "")
                fore = author.get("ForeName", "")
                if last:
                    author_names.append(f"{last} {fore}".strip())
            if len(article["AuthorList"]) > 3:
                author_names.append("et al.")
            authors = ", ".join(author_names)

        # Year — nested inside PubDate
        year = "Unknown"
        if "Journal" in article:
            journal_info = article["Journal"]
            if "JournalIssue" in journal_info:
                pub_date = journal_info["JournalIssue"].get("PubDate", {})
                year = str(pub_date.get("Year",
                       pub_date.get("MedlineDate", "Unknown")))

        # Journal name
        journal = "Unknown"
        if "Journal" in article:
            journal = str(article["Journal"].get("Title", "Unknown"))

        # Abstract — may have multiple sections
        # We join them into one paragraph
        abstract = "No abstract available"
        if "Abstract" in article:
            abstract_texts = article["Abstract"].get(
                "AbstractText", []
            )
            if isinstance(abstract_texts, list):
                # structured abstract with sections
                abstract = " ".join([
                    str(section) for section in abstract_texts
                ])
            else:
                # plain abstract
                abstract = str(abstract_texts)

        # PMID for reference
        pmid = str(record["MedlineCitation"]["PMID"])

        # Store as clean dictionary
        papers_list.append({
            "pmid": pmid,
            "title": title,
            "authors": authors,
            "year": year,
            "journal": journal,
            "abstract": abstract[:500]  # first 500 chars only
        })

    print(f"[Agent 3] ✅ {len(papers_list)} papers parsed successfully")

    # Block 9: auto-save every fetched paper into persistent memory.
    # This is why memory grows every run, with no LLM involvement.
    store_papers_in_memory(papers_list)

    # ── STEP 4: Save to whiteboard with duplicate check ───
    # Extract PMIDs already stored so we don't add same paper twice
    # This handles overlap between two PubMed searches in same run
    existing_pmids = {p["pmid"] for p in _pubmed_results} if _pubmed_results else set()

    # Only keep papers whose PMID is not already stored
    new_papers = [p for p in papers_list if p["pmid"] not in existing_pmids]

    # Save to whiteboard
    if _pubmed_results is None:
        _pubmed_results = new_papers
    else:
        _pubmed_results += new_papers

    print(f"[Agent 3] ✅ {len(new_papers)} new papers added "
          f"({len(papers_list) - len(new_papers)} duplicates skipped)")

    # Return summary to GPT
    # We give GPT the titles and abstracts to read
    # so it can write an informed comparison later
    gpt_summary = ""
    for i, paper in enumerate(papers_list):
        gpt_summary += (
            f"Paper {i+1}: {paper['title']} "
            f"({paper['authors']}, {paper['year']})\n"
            f"Abstract: {paper['abstract']}\n\n"
        )

    return json.dumps({
        "status": "success",
        "papers_found": len(papers_list),
        "results": gpt_summary[:3000]  # limit for GPT context
    })


# ── AGENT 4: SEARCH WEB ───────────────────────────────────
def run_search_web(query):
    """
    Agent 4 - Searches web for general information
    Use only for non-medical queries
    For medical topics always use search_pubmed
    
    Args:
        query → general search term
    
    Returns:
        JSON with search results for GPT
    """
    print(f"\n[Agent 4] Searching web: {query}")
    
    results = tavily_client.search(query)
    
    # Combine all results into one text
    search_text = ""
    for result in results["results"]:
        search_text += f"Title: {result['title']}\n"
        search_text += f"Content: {result['content']}\n\n"
    
    return json.dumps({
        "status": "success",
        "results": search_text[:3000]
    })


# ── AGENT 5: GENERATE REPORT ──────────────────────────────
def run_generate_report(research_question):
    """
    Agent 5 - Creates Word document with all findings
    Must run LAST after all other agents complete

    IMPROVEMENTS IN THIS VERSION:
    - Published literature shown as proper Word table
    - Table columns: #, Title, Authors, Year, Journal, Key Finding
    - GPT writes a scientific comparison paragraph
      using only numbers explicitly found in the papers
    - GPT also extracts one key finding per paper for the table

    Args:
        research_question - original question you asked

    Returns:
        JSON with filename of saved Word document
    """
    print(f"\n[Agent 5] Generating Word report...")

    # Safety check — verify required results exist
    # any() returns True if ANY item in list is None
    # meaning that agent did not run yet
    if any(x is None for x in [
        _mutation_rate_results, _table1_results, _tmb_results
    ]):
        return json.dumps({
            "status": "error",
            "message": "Please run fetch_cbioportal_data "
                      "and analyze_mutations first"
        })

    # Create new empty Word document
    doc = Document()

    # ── TITLE ─────────────────────────────────────────────
    doc.add_heading("ESR1 Mutation Validation Report", level=1)
    doc.add_paragraph(
        f"Generated: {datetime.now().strftime('%d %B %Y %H:%M')}"
    )
    doc.add_paragraph(
        "Neha Bansal | Eli Lilly and Company | "
        "Manuscript in preparation, 2026"
    )
    doc.add_paragraph("─" * 60)

    # ── RESEARCH QUESTION ─────────────────────────────────
    doc.add_heading("Research Question", level=2)
    doc.add_paragraph(research_question)

    # ── STUDY INFORMATION ─────────────────────────────────
    doc.add_heading("Validation Dataset", level=2)
    doc.add_paragraph(
        f"Database: cBioPortal\n"
        f"Study: MSK Metastatic Breast Cancer "
        f"(Cancer Discovery 2022)\n"
        f"Study ID: {STUDY_ID}\n"
        f"Total patients: "
        f"{_mutation_rate_results['total_patients']}"
    )

    # ── KEY FINDINGS ──────────────────────────────────────
    doc.add_heading("Key Findings", level=2)
    doc.add_paragraph(
        f"ESR1 Mutation Positivity Rate: "
        f"{_mutation_rate_results['positivity_rate']}% "
        f"({_mutation_rate_results['patients_positive']} of "
        f"{_mutation_rate_results['total_patients']} patients)"
    )
    doc.add_paragraph(
        f"TMB Analysis:\n"
        f"  ESR1+ median TMB: {_tmb_results['positive_median']} "
        f"(IQR: {_tmb_results['positive_iqr']})\n"
        f"  ESR1- median TMB: {_tmb_results['negative_median']} "
        f"(IQR: {_tmb_results['negative_iqr']})\n"
        f"  P-value: {_tmb_results['p_value']} "
        f"({'Significant' if _tmb_results['significant'] else 'Not significant'})"
    )

    # ── PUBLISHED LITERATURE TABLE ─────────────────────────
    doc.add_heading("Published Literature Review", level=2)
    doc.add_paragraph(
        f"PubMed search identified {len(_pubmed_results) if _pubmed_results else 0} "
        f"papers relevant to ESR1 mutation in metastatic breast cancer."
    )

    if _pubmed_results:

        # ── STEP 1: Ask GPT to extract key finding per paper ─
        # We send all abstracts to GPT and ask it to return
        # one short key finding sentence per paper
        # This fills the last column of our Word table
        print("[Agent 5] Asking GPT to extract key findings...")

        # Build a numbered list of abstracts for GPT to read
        abstracts_for_gpt = ""
        for i, paper in enumerate(_pubmed_results):
            abstracts_for_gpt += (
                f"Paper {i+1}: {paper['title']}\n"
                f"Abstract: {paper['abstract']}\n\n"
            )

        # Ask GPT to extract one key finding per paper
        # We ask for JSON so we can parse it reliably
        extraction_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{
                "role": "user",
                "content": (
                    f"For each paper below extract ONE key finding "
                    f"as a single sentence of maximum 20 words.\n\n"
                    f"{abstracts_for_gpt[:4000]}\n\n"
                    f"Return ONLY a JSON array like this:\n"
                    f'[{{"paper": 1, "finding": "..."}}, '
                    f'{{"paper": 2, "finding": "..."}}]\n'
                    f"No other text. No markdown. Just the JSON array."
                )
            }],
            max_tokens=800
        )

        # Parse GPT's JSON response into a list of findings
        # If parsing fails use a fallback value
        try:
            findings_text = extraction_response.choices[0].message.content
            # strip any accidental markdown backticks
            findings_text = findings_text.replace("```json", "").replace("```", "").strip()
            findings_list = json.loads(findings_text)
            # convert to a simple dictionary: {1: "finding", 2: "finding"}
            findings_dict = {
                item["paper"]: item["finding"]
                for item in findings_list
            }
        except Exception:
            # if GPT returns something unparseable use fallback
            findings_dict = {
                i+1: "See abstract for details"
                for i in range(len(_pubmed_results))
            }

        # ── STEP 2: Build the Word table ──────────────────
        # 6 columns: #, Title, Authors, Year, Journal, Key Finding
        # First row is the header row
        print("[Agent 5] Building literature table...")

        # add_table(rows, cols) creates empty table
        # rows=1 because we add data rows in the loop below
        # cols=6 for our six columns
        table = doc.add_table(rows=1, cols=6)
        table.style = "Table Grid"  # adds visible borders

        # Set column header text
        # table.rows[0].cells gives us the header row cells
        headers = ["#", "Title", "Authors", "Year", "Journal", "Key Finding"]
        header_cells = table.rows[0].cells
        for i, header in enumerate(headers):
            header_cells[i].text = header
            # make header text bold
            header_cells[i].paragraphs[0].runs[0].bold = True

        # Add one row per paper
        for i, paper in enumerate(_pubmed_results):
            # add_row() adds a new empty row at the bottom
            row = table.add_row()
            cells = row.cells

            # fill each cell with the paper's data
            cells[0].text = str(i + 1)                          # paper number
            cells[1].text = paper["title"][:150]                # title, max 150 chars
            cells[2].text = paper["authors"]                    # first 3 authors + et al
            cells[3].text = paper["year"]                       # publication year
            cells[4].text = paper["journal"][:50]               # journal name, max 50 chars
            cells[5].text = findings_dict.get(i + 1,            # GPT key finding
                           "See abstract for details")          # fallback if missing

        # Set column widths for readability
        # Total Word page width is ~6 inches with margins
        # We distribute across 6 columns
        from docx.shared import Inches
        col_widths = [
            Inches(0.3),   # # — narrow number column
            Inches(1.8),   # Title — widest column
            Inches(1.2),   # Authors
            Inches(0.4),   # Year — narrow
            Inches(1.0),   # Journal
            Inches(1.3)    # Key Finding
        ]
        for row in table.rows:
            for i, cell in enumerate(row.cells):
                cell.width = col_widths[i]

        doc.add_paragraph("")  # blank line after table

        # ── STEP 3: GPT comparison paragraph ──────────────
        # Now ask GPT to write a scientific comparison
        # between our cBioPortal result and the literature
        print("[Agent 5] Asking GPT to write comparison paragraph...")

        # Build a clean summary of papers for GPT
        # We include title, year, and abstract for context
        papers_summary = ""
        for i, paper in enumerate(_pubmed_results):
            papers_summary += (
                f"Paper {i+1} ({paper['year']}): "
                f"{paper['title']}\n"
                f"Abstract: {paper['abstract']}\n"
                f"Key finding: {findings_dict.get(i+1, 'N/A')}\n\n"
            )

        comparison_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{
                "role": "user",
                "content": (
                    f"You are a scientific writer for a pharmaceutical "
                    f"real-world evidence manuscript.\n\n"
                    f"Our study found: ESR1 mutation positivity rate of "
                    f"{_mutation_rate_results['positivity_rate']}% "
                    f"({_mutation_rate_results['patients_positive']} of "
                    f"{_mutation_rate_results['total_patients']} patients) "
                    f"in metastatic breast cancer using cBioPortal MSK data.\n\n"
                    f"TMB findings: ESR1+ median TMB "
                    f"{_tmb_results['positive_median']} vs "
                    f"ESR1- median TMB {_tmb_results['negative_median']}, "
                    f"p={_tmb_results['p_value']}.\n\n"
                    f"Published papers for comparison:\n{papers_summary[:4000]}\n\n"
                    f"Write a 4-5 sentence scientific comparison paragraph that:\n"
                    f"1. States the ESR1 positivity rates reported in the literature\n"
                    f"2. Compares our finding of "
                    f"{_mutation_rate_results['positivity_rate']}% "
                    f"against those rates\n"
                    f"3. Comments on consistency or discrepancy with reasons\n"
                    f"4. Notes any TMB findings from the literature if present\n"
                    f"5. Concludes with a statement on real-world validity\n\n"
                    f"CRITICAL RULES:\n"
                    f"- Only use numbers explicitly stated in the papers above\n"
                    f"- If no specific ESR1 positivity rates are stated say: "
                    f"'The identified literature did not report specific ESR1 "
                    f"positivity rates for direct numerical comparison.'\n"
                    f"- Do not infer, estimate, or speculate\n"
                    f"- Do not make up any statistics\n"
                    f"- Write in third person scientific style\n"
                    f"- Maximum 150 words"
                )
            }],
            max_tokens=300
        )

        comparison_text = comparison_response.choices[0].message.content
        doc.add_heading("Comparison with Published Literature", level=3)
        doc.add_paragraph(comparison_text)

    else:
        # No PubMed results — note clearly
        doc.add_paragraph(
            "No published literature found for comparison. "
            "Please run search_pubmed tool first."
        )

    # ── TMB VISUALIZATION ─────────────────────────────────
    # ── TMB VISUALIZATION ─────────────────────────────────
    # Use absolute path to ensure file is found regardless
    # of which directory Python is currently running from
    # os.path.join(os.getcwd()) converts relative to absolute
    # e.g. "tmb_ESR1_comparison.png" becomes
    # "C:\Users\bansa\Downloads\AI\tmb_ESR1_comparison.png"
    # This was the bug — relative path not resolving correctly
    plot_path = os.path.join(os.getcwd(), _plot_file)
    
    if plot_path and os.path.exists(plot_path):
        doc.add_heading("TMB Visualization", level=2)
        doc.add_picture(plot_path, width=Inches(6))
    else:
        doc.add_heading("TMB Visualization", level=2)
        doc.add_paragraph(
            f"Plot file not found at: {plot_path}. "
            f"Please ensure tmb_ESR1_comparison.png exists "
            f"in {os.getcwd()}"
        )

    # ── DEMOGRAPHICS SUMMARY ──────────────────────────────
    doc.add_heading("Patient Demographics Summary", level=2)
    doc.add_paragraph(
        f"Total patients: {_table1_results['total_patients']}\n"
        f"Mutation positive: {_table1_results['mutation_positive']}\n"
        f"Mutation negative: {_table1_results['mutation_negative']}\n"
        f"Median age: {_table1_results['median_age']}\n"
        f"Female: {_table1_results['pct_female']}%\n"
        f"White: {_table1_results['pct_white']}%\n"
        f"Top metastatic site: "
        f"{_table1_results['top_metastatic_site']}"
    )

    # ── DISCLAIMER ────────────────────────────────────────
    doc.add_paragraph("─" * 60)
    doc.add_paragraph(
        "Generated by ESR1 Validation Agent | "
        "Always verify findings independently "
        "before use in publications or protocols."
    )

    # ── SAVE FILE ─────────────────────────────────────────
    filename = (
        f"ESR1_Validation_Report_"
        f"{datetime.now().strftime('%Y%m%d_%H%M')}.docx"
    )
    doc.save(filename)

    print(f"[Agent 5] ✅ Report saved as: {filename}")

    return json.dumps({
        "status": "success",
        "filename": filename,
        "message": f"Report saved as {filename}"
    })

# ── TOOL DISPATCHER ───────────────────────────────────────
# Routes GPT tool calls to the correct Python function
# Like a switchboard operator connecting calls
# GPT says which tool - dispatcher runs the right function
def dispatch_tool(tool_name, tool_input):
    """
    Routes GPT tool calls to correct Python function

    Args:
        tool_name  - which tool GPT wants to use
        tool_input - parameters GPT wants to pass

    Returns:
        result from the tool as a JSON string
    """
    if tool_name == "fetch_cbioportal_data":
        return run_fetch_cbioportal_data(
            tool_input["study_id"],
            tool_input["gene_symbol"],
            tool_input["entrez_gene_id"]
        )
    elif tool_name == "analyze_mutations":
        return run_analyze_mutations(
            tool_input["gene_symbol"]
        )
    elif tool_name == "search_memory":
        return run_search_memory(tool_input["query"])
    elif tool_name == "search_pubmed":
        return run_search_pubmed(tool_input["query"])
    elif tool_name == "search_web":
        return run_search_web(tool_input["query"])
    elif tool_name == "generate_report":
        return run_generate_report(
            tool_input["research_question"]
        )
    else:
        return json.dumps({
            "status": "error",
            "message": f"Unknown tool: {tool_name}"
        })


print("✅ All tool execution functions ready!")
print("\nWhiteboard variables initialized:")
print("→ _patient_data          = None")
print("→ _sample_data           = None")
print("→ _gene_mutations        = None")
print("→ _merged_data           = None")
print("→ _mutation_rate_results = None")
print("→ _table1_results        = None")
print("→ _tmb_results           = None")
print("→ _plot_file             = None")
print("→ _pubmed_results        = None")
print("\nTool dispatcher ready for 5 agents!")

✅ All tool execution functions ready!

Whiteboard variables initialized:
→ _patient_data          = None
→ _sample_data           = None
→ _gene_mutations        = None
→ _merged_data           = None
→ _mutation_rate_results = None
→ _table1_results        = None
→ _tmb_results           = None
→ _plot_file             = None
→ _pubmed_results        = None

Tool dispatcher ready for 5 agents!


In [ ]:
import json

print(dispatch_tool("search_memory", {"query": "ESR1 mutation breast cancer"}))


[Agent 6] Searching memory: ESR1 mutation breast cancer
[Agent 6] Memory is empty — nothing stored yet.


NameError: name 'json' is not defined

In [1]:
# ============================================================
# BLOCK 9 - Part 1: Persistent memory + search_memory tool
# ============================================================
# Connects your ChromaDB RAG work to the ESR1 agent. Adds a 6th
# capability: the agent REMEMBERS papers across runs instead of
# re-fetching from PubMed every time.
#
# KEY DESIGN DECISIONS:
# - DEDICATED collection "esr1_agent_memory", separate from your toy
#   "rwe_literature_openai" playground. Starts EMPTY. Real papers only.
# - Papers retrieved from memory are written into _pubmed_results, the
#   SAME whiteboard Agent 5 reads, so they reach the report.
# - Storing is automatic (Part 2), not a tool GPT decides to call.
# ============================================================

import os
import getpass
import chromadb
from chromadb.utils import embedding_functions

# --- 1. Embedding function (MUST match what you stored with) ---------
# Reuse your existing OpenAI key if it's in the environment; else prompt.
_openai_key = os.environ.get("OPENAI_API_KEY")
if not _openai_key:
    _openai_key = getpass.getpass("OpenAI API key (for memory embeddings): ")

# Same model as Block 8 — text-embedding-3-small (1536 dims).
# Storing and querying MUST use the same model or distances are noise.
memory_embedder = embedding_functions.OpenAIEmbeddingFunction(
    api_key=_openai_key,
    model_name="text-embedding-3-small"
)

# --- 2. Connect to the persistent store ------------------------------
# Same chroma_db folder. PersistentClient = memory survives kernel
# restarts, which is what makes "remember across runs" real.
memory_client = chromadb.PersistentClient(path="chroma_db")

# --- 3. DEDICATED collection for the agent's real memory -------------
# New name on purpose. Empty on first run. Fills with real PubMed papers.
memory_collection = memory_client.get_or_create_collection(
    name="esr1_agent_memory",
    embedding_function=memory_embedder
)

# --- 4. Tunable relevance threshold + a log for tuning it later -------
# Vector search ALWAYS returns nearest neighbours, even bad ones. This
# cutoff turns "nearest" into "actually relevant": if the closest match
# is farther than this, treat memory as having nothing.
# 1.15 is a STARTING GUESS for OpenAI distances — tune it from the log
# below once real papers flow. Lower = stricter (more PubMed, safer);
# higher = more lenient (trusts memory more).
MEMORY_DISTANCE_THRESHOLD = 1.15

# Every search_memory call appends its distances here. After a few real
# runs, inspect _memory_distance_log to see where relevant vs irrelevant
# papers separate, then set the threshold properly.
_memory_distance_log = []


# --- 5. STORE helper: write PubMed papers into memory ----------------
# Called automatically from run_search_pubmed (Part 2). Uses UPSERT, not
# add() — upsert overwrites by ID instead of erroring on duplicates, so
# the same PMID across runs never crashes or double-stores. This is the
# clean fix for the duplicate problem you hit earlier.
def store_papers_in_memory(papers):
    """
    Saves a list of paper dicts into the persistent memory collection.

    Args:
        papers - list of dicts shaped like _pubmed_results entries:
                 {pmid, title, authors, year, journal, abstract}
    """
    if not papers:
        return  # nothing to store

    # ChromaDB needs three parallel lists: ids, documents, metadatas.
    # id        = PMID  (unique key — upsert dedups on this)
    # document  = abstract (this is what gets embedded into a vector)
    # metadata  = the other fields, to rebuild the paper on retrieval
    #             (metadata values MUST be str/int/float/bool — never None)
    ids        = [str(p["pmid"]) for p in papers]
    documents  = [str(p["abstract"]) for p in papers]
    metadatas  = [{
        "pmid":    str(p["pmid"]),
        "title":   str(p["title"])[:250],
        "authors": str(p["authors"])[:250],
        "year":    str(p["year"]),
        "journal": str(p["journal"])[:120],
    } for p in papers]

    # upsert = insert new, overwrite existing-by-id. Idempotent and safe.
    memory_collection.upsert(
        ids=ids,
        documents=documents,
        metadatas=metadatas
    )
    print(f"[Memory] Stored/updated {len(papers)} papers "
          f"(memory now holds {memory_collection.count()} total)")


# --- 6. AGENT 6: SEARCH MEMORY ---------------------------------------
# The new tool GPT can call. Queries memory FIRST, before PubMed.
# Relevant hits are written into _pubmed_results so the report agent
# picks them up — that's the integration seam.
def run_search_memory(query):
    """
    Agent 6 - Searches persistent memory for relevant papers.

    Args:
        query - medical search term, same style as search_pubmed

    Returns:
        JSON summary for GPT (how many relevant papers were found)
    """
    global _pubmed_results, _memory_distance_log

    print(f"\n[Agent 6] Searching memory: {query}")

    # Guard: empty memory (first ever run). Short-circuit to PubMed.
    stored_count = memory_collection.count()
    if stored_count == 0:
        print("[Agent 6] Memory is empty — nothing stored yet.")
        return json.dumps({
            "status": "no_relevant_memory",
            "papers_in_memory": 0,
            "relevant_found": 0,
            "message": "Memory is empty. Call search_pubmed to fetch papers."
        })

    # Ask for up to 5 nearest neighbours, but never more than we have
    # (some ChromaDB versions error if n_results > collection size).
    n = min(5, stored_count)
    results = memory_collection.query(query_texts=[query], n_results=n)

    # query returns lists-of-lists (one inner list per query text).
    # We sent one query, so take index [0] of each.
    ids   = results["ids"][0]
    docs  = results["documents"][0]
    metas = results["metadatas"][0]
    dists = results["distances"][0]

    # Log every distance for later threshold tuning.
    for d in dists:
        _memory_distance_log.append({"query": query, "distance": float(d)})

    # Apply the threshold. Keep only papers CLOSER than the cutoff.
    relevant = []
    for id_, doc, meta, dist in zip(ids, docs, metas, dists):
        print(f"[Agent 6]   distance={dist:.4f} | {meta.get('title','')[:60]}...")
        if dist <= MEMORY_DISTANCE_THRESHOLD:
            # Rebuild a paper dict in the EXACT shape _pubmed_results uses,
            # so the report agent treats memory papers like PubMed papers.
            relevant.append({
                "pmid":     meta.get("pmid", id_),
                "title":    meta.get("title", "No title available"),
                "authors":  meta.get("authors", "Unknown"),
                "year":     meta.get("year", "Unknown"),
                "journal":  meta.get("journal", "Unknown"),
                "abstract": doc,
            })

    # Nothing cleared the threshold → tell GPT to fetch from PubMed.
    if not relevant:
        print("[Agent 6] No papers passed the relevance threshold.")
        return json.dumps({
            "status": "no_relevant_memory",
            "papers_in_memory": stored_count,
            "relevant_found": 0,
            "message": "No sufficiently relevant papers in memory. "
                       "Call search_pubmed."
        })

    # --- Write relevant memory papers into the report whiteboard ------
    # Dedup against whatever is already on _pubmed_results, by PMID.
    existing_pmids = {p["pmid"] for p in _pubmed_results} if _pubmed_results else set()
    new_from_memory = [p for p in relevant if p["pmid"] not in existing_pmids]

    if _pubmed_results is None:
        _pubmed_results = new_from_memory
    else:
        _pubmed_results += new_from_memory

    print(f"[Agent 6] ✅ {len(relevant)} relevant papers found, "
          f"{len(new_from_memory)} added to report whiteboard")

    return json.dumps({
        "status": "success",
        "papers_in_memory": stored_count,
        "relevant_found": len(relevant),
        "message": f"Found {len(relevant)} relevant papers already in memory."
    })


print("✅ Block 9 Part 1 ready — memory_collection + search_memory live")
print(f"   Memory currently holds {memory_collection.count()} papers")

✅ Block 9 Part 1 ready — memory_collection + search_memory live
   Memory currently holds 0 papers


In [23]:
# ============================================================
# CELL 8 - System Prompt for ESR1 Validation Agent
# ============================================================
# This tells GPT exactly how to behave as a
# validation agent for RWE studies
#
# KEY INSTRUCTIONS:
# 1. Always fetch data first before analyzing
# 2. Always search PubMed for published comparisons
# 3. Always generate a report at the end
# 4. Never make up numbers or findings
# 5. Always follow a specific order of steps
#
# WHY ORDER MATTERS:
# fetch_cbioportal_data → must run first
# analyze_mutations     → needs data from step 1
# search_pubmed         → can run in parallel with step 2
# generate_report       → must run last
# ============================================================

VALIDATION_SYSTEM_PROMPT = """
You are an expert RWE Validation Agent specializing in 
cancer genomics and real world evidence research.

AVAILABLE STUDIES - always choose from this list only.
Use the study_id exactly as written — no modifications.

Metastatic Breast Cancer:
  study_id: breast_ink4_msk_2021
  description: MSK Metastatic Breast Cancer, Cancer Discovery 2022
  best for: ESR1, PIK3CA, TP53, CDH1 in metastatic setting

Early/Primary Breast Cancer:
  study_id: brca_metabric
  description: METABRIC primary breast cancer cohort
  best for: TP53, PIK3CA, CDH1 in early/primary setting

TCGA Pan-Cancer Breast:
  study_id: brca_tcga_pan_can_atlas_2018
  description: TCGA PanCancer Atlas breast cancer
  best for: broad genomic landscape, any breast cancer gene

MSK Breast Cancer 2018:
  study_id: breast_msk_2018
  description: MSK breast cancer Cancer Cell 2018
  best for: treatment resistance mutations

NSCLC Lung Cancer:
  study_id: luad_tcga_pan_can_atlas_2018
  description: TCGA Lung Adenocarcinoma PanCancer Atlas
  best for: KRAS, EGFR, ALK in lung adenocarcinoma

Prostate Cancer:
  study_id: prad_tcga_pan_can_atlas_2018
  description: TCGA Prostate PanCancer Atlas
  best for: PTEN, TP53, AR in prostate cancer

Colorectal Cancer:
  study_id: coadread_tcga_pan_can_atlas_2018
  description: TCGA Colorectal PanCancer Atlas
  best for: KRAS, APC, TP53 in colorectal cancer

STUDY SELECTION RULES:
- Mentions metastatic breast cancer → breast_ink4_msk_2021
- Mentions early or primary breast cancer → brca_metabric
- Mentions lung or NSCLC → luad_tcga_pan_can_atlas_2018
- Mentions prostate → prad_tcga_pan_can_atlas_2018
- Mentions colorectal or colon → coadread_tcga_pan_can_atlas_2018
- Mentions TCGA or pan-cancer → use relevant _tcga_pan_can_atlas_2018
- If unsure → default to breast_ink4_msk_2021

YOUR JOB:
Validate mutation findings from proprietary databases 
like Flatiron against independent public datasets 
from cBioPortal, supported by published literature.

STRICT TOOL ORDER - always follow this sequence:
Step 1 - fetch_cbioportal_data
         Always start here
         Fetch patient, sample and mutation data
         
Step 2 - analyze_mutations  
         Run after Step 1
         Calculate positivity rates
         Generate Table 1 and TMB analysis
         
Step 3 - search_pubmed
         MUST be called TWICE with two different queries
         Search 1 (broad): gene name + cancer type only
                           e.g. "ESR1 mutation metastatic breast cancer"
         Search 2 (treatment focused): gene + resistance + treatment
                           e.g. "ESR1 endocrine resistance breast cancer treatment"
         Never add "real world", "ER-positive", "HER2-negative" to queries
         Broad queries return more papers — always prefer broad
         Both searches must complete before calling generate_report
         Always use search_pubmed for medical topics
         Never use search_web for medical research
         
Step 4 - generate_report
         YOU MUST CALL THIS TOOL BEFORE WRITING ANY RESPONSE
         DO NOT write a summary until generate_report has run
         DO NOT say the report will be generated — CALL THE TOOL
         The Word document does not exist until you call this tool

CRITICAL RULE:
Your final text response must come AFTER generate_report runs.
If you have not called generate_report yet, call it NOW.
Do not produce a final answer without calling generate_report first.

RULES:
- Never make up mutation rates or statistics
- Only report what tools actually return
- Always compare cBioPortal findings against
  published literature found in PubMed
- Flag clearly if findings are consistent
  or inconsistent with published data
- Use search_web only for non-medical queries

OUTPUT FORMAT for your final answer:
1. cBioPortal findings summary
2. Published literature comparison  
3. Consistency assessment
4. Key clinical insights
5. Limitations
6. Report file location
"""

print("✅ Validation system prompt ready!")
print(f"\nPrompt length: {len(VALIDATION_SYSTEM_PROMPT)} characters")
print("\nAgent will follow this strict tool order:")
print("Step 1 - fetch_cbioportal_data")
print("Step 2 - analyze_mutations")
print("Step 3 - search_pubmed")
print("Step 4 - generate_report")
print("\nAvailable studies:")
print("- breast_ink4_msk_2021    (metastatic breast cancer)")
print("- brca_metabric           (early breast cancer)")
print("- brca_tcga_pan_can_atlas_2018 (TCGA breast)")
print("- breast_msk_2018         (MSK breast 2018)")
print("- luad_tcga_pan_can_atlas_2018 (NSCLC)")
print("- prad_tcga_pan_can_atlas_2018 (prostate)")
print("- coadread_tcga_pan_can_atlas_2018 (colorectal)")

✅ Validation system prompt ready!

Prompt length: 3922 characters

Agent will follow this strict tool order:
Step 1 - fetch_cbioportal_data
Step 2 - analyze_mutations
Step 3 - search_pubmed
Step 4 - generate_report

Available studies:
- breast_ink4_msk_2021    (metastatic breast cancer)
- brca_metabric           (early breast cancer)
- brca_tcga_pan_can_atlas_2018 (TCGA breast)
- breast_msk_2018         (MSK breast 2018)
- luad_tcga_pan_can_atlas_2018 (NSCLC)
- prad_tcga_pan_can_atlas_2018 (prostate)
- coadread_tcga_pan_can_atlas_2018 (colorectal)


In [24]:
# ============================================================
# CELL 9 - ESR1 Validation Agent Loop
# ============================================================
# The heart of the agent system
# Keeps running until GPT completes all 4 steps
# and generates the final report
#
# HOW IT WORKS:
# 1. Send research question to GPT
# 2. GPT decides which tool to call first
# 3. We run that tool and return results
# 4. GPT reads results and decides next tool
# 5. Repeat until GPT says it is done
# 6. GPT gives final structured answer
#
# SAFETY:
# max_steps = 15 prevents infinite loops
# Each tool call counted as one step
# If GPT has not finished in 15 steps
# we force stop and return what we have
#
# CONVERSATION HISTORY:
# Full history sent to GPT every step
# So GPT remembers what it already did
# and what results came back
# Same concept as Phase 1 chatbot
# just with tools added on top
# ============================================================

def run_validation_agent(research_question):
    """
    Main ESR1 Validation Agent function.
    Takes a research question and runs all 4 steps
    automatically using GPT to orchestrate.
    
    Args:
        research_question → your validation question
                            e.g. "Validate ESR1 mutation 
                            findings in metastatic breast cancer"
    
    Returns:
        GPT's final structured validation summary
        plus a saved Word document report
    """
    
    print("=" * 60)
    print("ESR1 VALIDATION AGENT")
    print("=" * 60)
    print(f"Research Question: {research_question}")
    print("=" * 60)
    
    # Initialize conversation history
    # Same concept as Phase 1 chatbot
    conversation_history = []
    
    # Add research question to history
    conversation_history.append({
        "role": "user",
        "content": research_question
    })
    
    # Safety limit — prevents infinite loops
    max_steps = 15
    current_step = 0
    
    # ── THE AGENT LOOP ────────────────────────────────────
    while current_step < max_steps:
        
        current_step += 1
        print(f"\n{'─' * 40}")
        print(f"Agent Step {current_step} of max {max_steps}")
        print(f"{'─' * 40}")
        
        # Send full history to GPT with all tools available
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": VALIDATION_SYSTEM_PROMPT
                },
                *conversation_history
            ],
            tools=tools  # all 5 tools available
        )
        
        gpt_response = response.choices[0].message
        
        # ── DID GPT WANT TO USE A TOOL? ──────────────────
        if gpt_response.tool_calls:
            
            # Add GPT tool request to history
            conversation_history.append({
                "role": "assistant",
                "content": str(gpt_response.content or ""),
                "tool_calls": gpt_response.tool_calls
            })
            
            # Run ALL tools GPT requested
            # GPT might request multiple at once
            for tool_call in gpt_response.tool_calls:
                
                tool_name = tool_call.function.name
                tool_input = json.loads(tool_call.function.arguments)
                
                print(f"\n[GPT] Calling tool: {tool_name}")
                print(f"[GPT] With inputs: {tool_input}")
                
                # Run the tool via dispatcher
                tool_result = dispatch_tool(tool_name, tool_input)
                
                print(f"[Tool] Result received ✅")
                
                # Add tool result to history
                conversation_history.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": tool_result
                })
        
        else:
            # ── GPT GAVE FINAL ANSWER ─────────────────────
            # No more tool calls — GPT is done
            final_answer = gpt_response.content
            
            # Add to history
            conversation_history.append({
                "role": "assistant",
                "content": final_answer
            })
            
            print(f"\n{'=' * 60}")
            print(f"✅ Agent completed in {current_step} steps!")
            print(f"{'=' * 60}")
            
            return final_answer
    
    # Max steps reached
    return "Agent reached maximum steps without completing. Please try again."


print("✅ Validation agent loop ready!")
print("\nUsage:")
print('run_validation_agent("Validate ESR1 mutation findings')
print(' in ER+/HER2- metastatic breast cancer")')

✅ Validation agent loop ready!

Usage:
run_validation_agent("Validate ESR1 mutation findings
 in ER+/HER2- metastatic breast cancer")


In [21]:
# Check which functions are defined
print("dispatch_tool defined:", "dispatch_tool" in dir())
print("run_fetch_cbioportal_data defined:", "run_fetch_cbioportal_data" in dir())
print("run_generate_report defined:", "run_generate_report" in dir())

dispatch_tool defined: False
run_fetch_cbioportal_data defined: True
run_generate_report defined: True


In [25]:
# ── RESET WHITEBOARD BEFORE EACH RUN ─────────────────────
# Clears all results from any previous run
# Always run this before starting a fresh agent run
_patient_data = None
_sample_data = None
_gene_mutations = None
_merged_data = None
_mutation_rate_results = None
_table1_results = None
_tmb_results = None
_plot_file = None
_pubmed_results = None
print("✅ Whiteboard reset — ready for fresh run")

✅ Whiteboard reset — ready for fresh run


In [26]:
# ============================================================
# CELL 10 - Run the Full Validation Agent
# ============================================================
# This single function call triggers all 5 agents:
# 1. Fetches cBioPortal data
# 2. Runs mutation analysis
# 3. Searches PubMed for published rates
# 4. Generates comparison
# 5. Creates Word report
#
# Everything happens automatically
# GPT orchestrates all steps
# You just ask the question
# ============================================================

# Run the full validation agent
final_answer = run_validation_agent(
    """Validate ESR1 mutation positivity rates in 
    ER-positive HER2-negative metastatic breast cancer 
    using cBioPortal public data and compare against 
    published real world literature"""
)

# Print the final structured answer
print("\n" + "=" * 60)
print("FINAL VALIDATION SUMMARY")
print("=" * 60)
print(final_answer)

ESR1 VALIDATION AGENT
Research Question: Validate ESR1 mutation positivity rates in 
    ER-positive HER2-negative metastatic breast cancer 
    using cBioPortal public data and compare against 
    published real world literature

────────────────────────────────────────
Agent Step 1 of max 15
────────────────────────────────────────

[GPT] Calling tool: fetch_cbioportal_data
[GPT] With inputs: {'study_id': 'breast_ink4_msk_2021', 'gene_symbol': 'ESR1', 'entrez_gene_id': 2099}

[Agent 1] Fetching cBioPortal data...
[Agent 1] Study: breast_ink4_msk_2021
[Agent 1] Gene: ESR1
Fetching patient data for study: breast_ink4_msk_2021...
✅ Patient data fetched: 1116 patients
Fetching sample data for study: breast_ink4_msk_2021...
✅ Sample data fetched: 1116 unique patients
Fetching ESR1 mutations for study: breast_ink4_msk_2021...
✅ ESR1 mutations fetched: 305 mutation events
   Unique patients with ESR1: 238
Building merged dataset...
✅ Merged dataset built: 1116 patients
   ESR1 Positive: 23

C:\Users\bansa\AppData\Local\Temp\ipykernel_24232\3444564384.py:345: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(


✅ TMB plot saved as: tmb_ESR1_comparison.png
[Tool] Result received ✅

────────────────────────────────────────
Agent Step 3 of max 15
────────────────────────────────────────

[GPT] Calling tool: search_pubmed
[GPT] With inputs: {'query': 'ESR1 mutation metastatic breast cancer'}

[Agent 3] Searching PubMed: ESR1 mutation metastatic breast cancer
[Agent 3] ✅ 10 papers parsed successfully
[Agent 3] ✅ 10 new papers added (0 duplicates skipped)
[Tool] Result received ✅

[GPT] Calling tool: search_pubmed
[GPT] With inputs: {'query': 'ESR1 endocrine resistance breast cancer treatment'}

[Agent 3] Searching PubMed: ESR1 endocrine resistance breast cancer treatment
[Agent 3] ✅ 10 papers parsed successfully
[Agent 3] ✅ 7 new papers added (3 duplicates skipped)
[Tool] Result received ✅

────────────────────────────────────────
Agent Step 4 of max 15
────────────────────────────────────────

[GPT] Calling tool: generate_report
[GPT] With inputs: {'research_question': 'Validate ESR1 mutation pos